# Week 1 and Week 2: Sequential Decision-Making Agent  
## Hidden World State, Monte Carlo Beliefs, Expected Utility, and Value of Information

This notebook implements a two-part sequential decision-making model in a campus environment.

In **Week 1**, the environment is modeled as a hidden world state where students move across campus locations over time. Since the full state is uncertain, Monte Carlo simulation is used to generate many possible traces of the world.

In **Week 2**, a rational decision-making layer is added on top of those simulations. The agent estimates beliefs about whether locations are busy, computes expected utilities for possible actions, and determines whether it is worth paying for additional information before acting.

This notebook demonstrates:
- Hidden-state simulation
- Probabilistic belief estimation
- Expected utility reasoning
- Value of information analysis
- Rational action selection under uncertainty

In [1]:
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

## Part 1 — Week 1: Hidden World State and Monte Carlo Traces

This section defines the campus world used for simulation.

The environment includes:
- student categories,
- academic or activity programs,
- campus locations,
- time slots across the day,
- and preference structures that influence where students are likely to go.

These structures are the foundation for the hidden-state simulation. They do not make decisions yet; instead, they define the world in which decisions will later be made.

In [2]:
# ============================================================
# WEEK 1 (Corrected): Hidden world state + Monte Carlo traces
# ============================================================

STUDENT_TYPES = ["athlete", "copa", "regular"]

ATHLETE_PROGRAMS = [
    "Baseball", "TrackAndField", "Wrestling", "Soccer", "ESports",
    "Softball", "Lacrosse", "Basketball", "Volleyball", "Golf"
]

COPA_PROGRAMS = [
    "Dance", "MusicalTheater", "Acting", "Film", "Animation", "ScreenWriting"
]

LOCATIONS = [
    "LawrenceHall",
    "ThayerHall",
    "WestPenn",
    "StudentCenter",
    "Library",
    "VillagePark",
    "PlayHouse",
    "BoulevardStudios",
    "BoulevardAppartments",
    "OffCampus",
]

TIME_SLOTS = ["08:00", "10:00", "12:00", "14:00", "16:00", "18:00", "21:00"]
NUM_TIME_STEPS = len(TIME_SLOTS)

## Student Type Preferences and Program Influence

This section specifies how different student types tend to behave across time.

Each student type has location preferences that vary throughout the day. In addition, specific programs boost the likelihood of certain locations. For example, athletes in Track and Field are more likely to favor VillagePark, while COPA students in Dance are more likely to favor BoulevardStudios.

These preferences create structure in the hidden world and allow the simulation to produce realistic variation in student movement.

In [3]:
STUDENT_TYPE_PREFERENCES = {
    "athlete": [
        ["StudentCenter", "OffCampus"],               # 08:00
        ["StudentCenter", "ThayerHall"],                   # 10:00
        ["VillagePark", "LawrenceHall"],      # 12:00
        ["Library", "ThayerHall"],                      # 14:00
        ["LawrenceHall", "WestPenn"],                    # 16:00
        ["WestPenn", "OffCampus"],                 # 18:00 
        ["StudentCenter", "OffCampus"],                # 21:00         
        LOCATIONS                                       
    ],
    "copa": [
        ["LawrenceHall", "BoulevardStudios"],              # 08:00
        ["PlayHouse", "LawrenceHall"],              # 10:00
        ["LawrenceHall", "Library"],                   # 12:00
        ["LawrenceHall", "ThayerHall"],             # 14:00
        ["PlayHouse", "BoulevardAppartments"],                 # 16:00
        ["OffCampus", "BoulevardAppartments"],       # 18:00  
        ["StudentCenter", "OffCampus"],                # 21:00
        LOCATIONS                                       
    ],
    "regular": [LOCATIONS] * NUM_TIME_STEPS
}

PROGRAM_BOOST = {
    # athlete examples
    "TrackAndField": ["VillagePark"],
    "Soccer": ["VillagePark"],
    "Lacrosse": ["VillagePark"],
    "Baseball": ["VillagePark"],
    "Golf": ["VillagePark"],

    "Basketball": ["StudentCenter"],
    "Volleyball": ["StudentCenter"],
    "ESports": ["BoulevardAppartments", "StudentCenter"],

    # copa examples
    "Dance": ["BoulevardStudios"],
    "MusicalTheater": ["PlayHouse"],
    "Acting": ["PlayHouse"],
    "Film": ["PlayHouse", "BoulevardStudios"],
    "Animation": ["BoulevardStudios"],
    "ScreenWriting": ["Library"],
}

## Data Classes for Internal State and Hidden World State

This section defines the main data structures used in the simulation.

- `AgentInternalState` stores stable information about each student, such as ID, type, program, and friend relationships.
- `HiddenWorldState` represents the true campus positions of all students at a particular time step.
- `Student` combines the internal state with a current location and includes a method for choosing where to go.

These classes make the simulation easier to organize and interpret.

In [4]:
@dataclass
class AgentInternalState:
    sid: str
    stype: str
    program: str
    friends: List[str] = field(default_factory=list)

@dataclass
class HiddenWorldState:
    """Truth at one time step: where everyone is."""
    t: int
    positions: Dict[str, str]  # sid -> location

@dataclass
class Student:
    internal: AgentInternalState
    location: Optional[str] = None

    def choose_location(
        self,
        prev_positions: Dict[str, str],
        time_step: int,
        friend_boost: int = 2,
        rng: Optional[random.Random] = None,
    ) -> str:
        rng = rng or random

        base = list(STUDENT_TYPE_PREFERENCES[self.internal.stype][time_step])

        boosted: List[str] = []

        # Friend influence: repeat friends' last locations
        for fid in self.internal.friends:
            if fid in prev_positions:
                boosted.append(prev_positions[fid])

        # Program influence: boost program-preferred places
        boosted += PROGRAM_BOOST.get(self.internal.program, [])

        # Boost by repetition (simple probability hack)
        possible = base + boosted * friend_boost if boosted else base
        return rng.choice(possible)

## Simulating a Day in the Hidden World

This section simulates how students move across the campus during the day.

At each time step:
- the order of students is randomized,
- each student selects a location,
- the full world state is recorded.

The result is a trace of the day, where each element represents the hidden truth about where everyone is at one point in time.

This is important because the later decision-making model does not directly observe the entire world. Instead, it must reason from simulated possibilities.

In [5]:
def simulate_day(
    students: List[Student],
    time_steps: int = NUM_TIME_STEPS,
    seed: Optional[int] = None
) -> List[HiddenWorldState]:
    rng = random.Random(seed)
    trace: List[HiddenWorldState] = []
    prev_positions: Dict[str, str] = {}

    for t in range(time_steps):
        positions: Dict[str, str] = {}
        order = students[:]
        rng.shuffle(order)

        for s in order:
            loc = s.choose_location(prev_positions, time_step=t, rng=rng)
            s.location = loc
            positions[s.internal.sid] = loc

        trace.append(HiddenWorldState(t=t, positions=positions))
        prev_positions = positions

    return trace

## Consistency Checking and Monte Carlo Filtering

This section supports probabilistic reasoning by generating many possible hidden-world traces and keeping only those that match any observations.

Two functions are used:
- `is_consistent(...)` checks whether a trace matches observed evidence.
- `monte_carlo_simulation(...)` repeatedly simulates the day and filters traces based on consistency.

This creates a Monte Carlo approximation of the agent’s belief about the world. Instead of knowing exactly what is true, the agent keeps many plausible worlds and reasons from them probabilistically.

In [6]:
def is_consistent(trace: List[HiddenWorldState], observed: Optional[List[Dict]]) -> bool:
    """
    observed format: [{"student":"a0","time":1,"location":"StudentCenter"}, ...]
    """
    for obs in observed or []:
        sid, t, loc = obs["student"], obs["time"], obs["location"]
        if t < 0 or t >= len(trace):
            return False
        if sid not in trace[t].positions:
            return False
        if trace[t].positions[sid] != loc:
            return False
    return True

def monte_carlo_simulation(
    students_template: List[Student],
    num_traces: int = 1000,
    time_steps: int = NUM_TIME_STEPS,
    observed: Optional[List[Dict]] = None,
    seed: int = 7,
) -> List[List[HiddenWorldState]]:
    kept: List[List[HiddenWorldState]] = []
    rng = random.Random(seed)

    for _ in range(num_traces):
        # Fresh copy each trace (including program + friends)
        students = [
            Student(
                AgentInternalState(
                    sid=s.internal.sid,
                    stype=s.internal.stype,
                    program=s.internal.program,
                    friends=list(s.internal.friends),
                )
            )
            for s in students_template
        ]
        trace = simulate_day(students, time_steps=time_steps, seed=rng.randint(0, 10**9))
        if is_consistent(trace, observed):
            kept.append(trace)

    return kept

## Part 2 — Week 2: Decision-Theoretic Agent

This section adds rational decision-making on top of the Week 1 simulation.

The agent now has:
- a specific student type,
- a time at which a decision must be made,
- a sensing option for gathering information,
- and a utility-based framework for comparing actions.

The goal is no longer just to simulate the world, but to choose the action with the highest expected utility under uncertainty.

In [7]:
# ============================================================
# WEEK 2: Decision-theoretic agent (EU + VOI) on top of Week 1
# ============================================================

# --- Week 2 settings  ---
AGENT_TYPE = "athlete"
DECISION_TIME = "10:00"
SENSE_LOCATION = "StudentCenter"
INFO_COST = 2.0
BUSY_THRESHOLD = 0.18  # preferred default (18%)

# Actions: ALL locations
ACTIONS = list(LOCATIONS)
SENSE_ACTION = f"Check{SENSE_LOCATION}Crowd"

## Utility Model

This section defines how the agent evaluates actions.

The utility model is based on expected time cost. A location is more attractive when it:
- requires less walking,
- is expected to involve less waiting,
- and better matches the student’s type.

The model is expressed in negative minutes, so higher utility means lower total expected cost.

This utility structure is what allows the agent to compare different destination choices in a rational and interpretable way.

In [8]:
# --- Utility model (multi-attribute, simple & explainable) ---
# All numbers are in "minutes" (we maximize utility = negative minutes).

# Base walk time defaults (by type)
DEFAULT_WALK = {"athlete": 6, "copa": 7, "regular": 5}

# Per-location walk-time overrides (optional but helpful for realism)
# If not present, falls back to DEFAULT_WALK[type]
WALK_OVERRIDES: Dict[Tuple[str, str], int] = {
    ("athlete", "VillagePark"): 4,
    ("athlete", "StudentCenter"): 5,
    ("athlete", "Library"): 6,
    ("athlete", "OffCampus"): 10,
}

# Mismatch penalty defaults: "how annoying is it for this type to be here?"
DEFAULT_MISMATCH = {"athlete": 1, "copa": 1, "regular": 0}

# Per-location mismatch overrides
MISMATCH_OVERRIDES: Dict[Tuple[str, str], int] = {
    ("athlete", "VillagePark"): 0,
    ("athlete", "StudentCenter"): 0,
    ("athlete", "BoulevardStudios"): 3,   # less "fit" for athlete
    ("athlete", "PlayHouse"): 3,
    ("athlete", "OffCampus"): 4,
}

# Wait times: default busy/not-busy
DEFAULT_WAIT_BUSY = 12
DEFAULT_WAIT_NOT = 6

# Per-location wait overrides (StudentCenter tends to be a bottleneck)
WAIT_OVERRIDES: Dict[str, Tuple[int, int]] = {
    "StudentCenter": (16, 7),  # (busy, not busy)
    "Library": (10, 5),
    "VillagePark": (7, 4),
    "OffCampus": (2, 2),
}

## Utility Functions and Best Action Selection

This section implements the utility calculations.

The functions compute:
- walking time,
- mismatch penalty,
- expected waiting time,
- overall utility,
- expected utility for each action,
- and the best action under the current belief state.

At this point, the agent can rank actions if it already has beliefs about busyness. The next step is to derive those beliefs from the Week 1 traces.

In [9]:
def walk_time(student_type: str, location: str) -> int:
    return WALK_OVERRIDES.get((student_type, location), DEFAULT_WALK[student_type])

def mismatch_penalty(student_type: str, location: str) -> int:
    return MISMATCH_OVERRIDES.get((student_type, location), DEFAULT_MISMATCH[student_type])

def expected_wait(p_busy: float, location: str) -> float:
    busy, not_busy = WAIT_OVERRIDES.get(location, (DEFAULT_WAIT_BUSY, DEFAULT_WAIT_NOT))
    return p_busy * busy + (1 - p_busy) * not_busy

def utility(student_type: str, location: str, p_busy_loc: float) -> float:
    """
    Utility = negative total expected time/cost.
    Higher is better.
    """
    total_cost = (
        walk_time(student_type, location)
        + expected_wait(p_busy_loc, location)
        + mismatch_penalty(student_type, location)
    )
    return -float(total_cost)

def expected_utility(student_type: str, belief_busy: Dict[str, float], action_location: str) -> float:
    return utility(student_type, action_location, belief_busy[action_location])

def best_action(student_type: str, belief_busy: Dict[str, float]) -> Tuple[str, float, Dict[str, float]]:
    eus = {loc: expected_utility(student_type, belief_busy, loc) for loc in ACTIONS}
    a_star = max(eus, key=eus.get)
    return a_star, eus[a_star], eus

## Belief State Construction from Monte Carlo Traces

This section converts the Week 1 Monte Carlo traces into probabilities.

For each location and time step, the model estimates:

\[
P(Busy(L,t)=True)
\]

A location is considered busy when the fraction of students at that location exceeds the chosen threshold.

These probabilities are central to decision-making because the agent does not know with certainty whether a location is crowded. Instead, it makes decisions using probabilistic beliefs derived from simulation.

In [10]:
# --- Belief model: Busy(L,t) estimated from Week 1 traces ---
def belief_busy_from_traces(
    traces: List[List[HiddenWorldState]],
    time_step: int,
    threshold: float = BUSY_THRESHOLD
) -> Dict[str, float]:
    """
    belief_busy[L] = P(Busy(L,t)=True)
    Busy(L,t) is True if fraction of students at L at time t > threshold.
    """
    belief = {L: 0.0 for L in LOCATIONS}
    if not traces:
        return belief

    for tr in traces:
        positions = tr[time_step].positions
        n = max(1, len(positions))
        # count students per location
        counts: Dict[str, int] = {L: 0 for L in LOCATIONS}
        for loc in positions.values():
            if loc in counts:
                counts[loc] += 1

        for L in LOCATIONS:
            frac = counts[L] / n
            if frac > threshold:
                belief[L] += 1.0

    for L in LOCATIONS:
        belief[L] /= len(traces)
    return belief

## Partial Observability and Bayesian Updating

This section introduces partial observability.

The agent cannot directly observe whether the sensed location is busy. Instead, it can take a sensing action that returns a noisy observation:
- `ReportBusy`
- `ReportNotBusy`

Using Bayes’ rule, the prior probability of busyness is updated into a posterior probability after receiving an observation.

This is the key idea behind belief-state reasoning in partially observable environments: the agent does not observe the truth directly, but updates its beliefs based on evidence.

In [11]:
# --- Information gathering: sensor model for Busy(StudentCenter,t) ---
OBS = ["ReportBusy", "ReportNotBusy"]
LIKELIHOOD = {
    ("ReportBusy", True): 0.85,
    ("ReportNotBusy", True): 0.15,
    ("ReportBusy", False): 0.15,
    ("ReportNotBusy", False): 0.85,
}

def bayes_update_busy(p_busy: float, obs: str) -> float:
    p_obs_busy = LIKELIHOOD[(obs, True)]
    p_obs_not = LIKELIHOOD[(obs, False)]
    num = p_obs_busy * p_busy
    den = num + p_obs_not * (1 - p_busy)
    return p_busy if den == 0 else num / den

def p_obs_from_prior(p_busy: float, obs: str) -> float:
    return LIKELIHOOD[(obs, True)] * p_busy + LIKELIHOOD[(obs, False)] * (1 - p_busy)

## Expected Utility with Information and Value of Information

This section evaluates whether gathering information is worthwhile.

The agent compares:
- the best direct action without sensing,
- versus the expected utility of first sensing and then acting optimally after updating its beliefs.

The difference between these quantities is the **Value of Information (VOI)**.

If VOI is positive enough to outweigh the sensing cost, then information is worth paying for. Otherwise, the agent should act immediately.

In [12]:
def expected_utility_with_information(
    student_type: str,
    belief_busy: Dict[str, float],
    sense_location: str,
    info_cost: float
) -> float:
    """
    Compute EU of taking the sensing action first:
      EU(check) = Σ_o P(o) * max_a EU(a | updated belief after observing o) - cost
    Only updates belief about the sensed location.
    """
    p_busy_prior = belief_busy[sense_location]
    total = 0.0

    for obs in OBS:
        po = p_obs_from_prior(p_busy_prior, obs)
        p_busy_post = bayes_update_busy(p_busy_prior, obs)

        post_belief = dict(belief_busy)
        post_belief[sense_location] = p_busy_post

        _, eu_best_post, _ = best_action(student_type, post_belief)
        total += po * eu_best_post

    return total - info_cost

def value_of_information(
    student_type: str,
    belief_busy: Dict[str, float],
    sense_location: str,
    info_cost: float
) -> float:
    direct_action, eu_direct, _ = best_action(student_type, belief_busy)
    eu_check = expected_utility_with_information(student_type, belief_busy, sense_location, info_cost)
    return eu_check - eu_direct

## Rational Agent Class

This section packages the full reasoning process into a single agent.

The agent:
1. determines the relevant time step,
2. builds a belief state from Monte Carlo traces,
3. computes expected utilities for all direct actions,
4. computes the expected utility of gathering information,
5. compares both strategies,
6. and returns the rational choice.

This class integrates simulation, belief estimation, Bayesian updating, expected utility, and value of information into one decision-making pipeline.

In [13]:
@dataclass
class CampusRationalAgent:
    student_type: str
    time_slot: str
    students_template: List[Student]
    num_traces: int = 2000
    busy_threshold: float = BUSY_THRESHOLD
    sense_location: str = SENSE_LOCATION
    info_cost: float = INFO_COST
    observations: Optional[List[Dict]] = None
    seed: int = 7

    def time_step(self) -> int:
        if self.time_slot not in TIME_SLOTS:
            raise ValueError(f"time_slot must be one of {TIME_SLOTS}")
        return TIME_SLOTS.index(self.time_slot)

    def build_belief(self) -> Dict[str, float]:
        traces = monte_carlo_simulation(
            students_template=self.students_template,
            num_traces=self.num_traces,
            time_steps=NUM_TIME_STEPS,
            observed=self.observations,
            seed=self.seed
        )
        return belief_busy_from_traces(traces, time_step=self.time_step(), threshold=self.busy_threshold)

    def choose(self) -> Dict[str, object]:
        belief = self.build_belief()

        a_direct, eu_direct, eus = best_action(self.student_type, belief)
        eu_check = expected_utility_with_information(self.student_type, belief, self.sense_location, self.info_cost)
        voi = eu_check - eu_direct

        if eu_check > eu_direct:
            choice = SENSE_ACTION
        else:
            choice = a_direct

        return {
            "belief_busy": belief,
            "eus": eus,
            "best_direct": (a_direct, eu_direct),
            "eu_check": eu_check,
            "voi": voi,
            "choice": choice
        }

## Demo: Building Students and Running the Agent

This final code section constructs a sample campus population and runs the rational agent.

The demo:
- creates athlete, COPA, and regular students,
- assigns programs and social links,
- builds the Monte Carlo belief state,
- computes expected utilities,
- evaluates the value of information,
- and prints the final choice.

This serves as the complete end-to-end example of the model.

In [14]:
# ============================================================
# DEMO (Week 2): build students, compute belief, EU, VOI, choose
# ============================================================

if __name__ == "__main__":
    random.seed(7)

    # --- Build students template  ---
    students: List[Student] = []

    # Athletes a0..a2
    for i in range(3):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"a{i}",
                    stype="athlete",
                    program=random.choice(ATHLETE_PROGRAMS),
                    friends=[f"a{(i+1)%3}"],
                )
            )
        )

    # COPA c0..c1
    for i in range(2):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"c{i}",
                    stype="copa",
                    program=random.choice(COPA_PROGRAMS),
                    friends=[f"c{(i+1)%2}"],
                )
            )
        )

    # Regular r0..r4
    for i in range(5):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"r{i}",
                    stype="regular",
                    program="None",
                    friends=[],
                )
            )
        )

    agent = CampusRationalAgent(
        student_type=AGENT_TYPE,
        time_slot=DECISION_TIME,
        students_template=students,
        num_traces=2000,
        busy_threshold=BUSY_THRESHOLD,
        sense_location=SENSE_LOCATION,
        info_cost=INFO_COST,
        observations=None,
        seed=7
    )

    result = agent.choose()
    belief_busy = result["belief_busy"]
    eus = result["eus"]
    a_direct, eu_direct = result["best_direct"]
    eu_check = result["eu_check"]
    voi = result["voi"]
    choice = result["choice"]

    t = agent.time_step()
    print(f"=== Week 2 Decision @ {DECISION_TIME} (time_step={t}) for agent type='{AGENT_TYPE}' ===")
    print(f"Uncertain variable: Busy(L,t) defined as frac(students at L) > {BUSY_THRESHOLD:.2f}")
    print(f"Sense action: {SENSE_ACTION} with cost={INFO_COST:.1f} (minutes)\n")

    # Show beliefs for each location
    print("=== Belief state (derived from Week 1 Monte Carlo) ===")
    for L in LOCATIONS:
        print(f"P(Busy({L}) at {DECISION_TIME}) ≈ {belief_busy[L]:.3f}")

    # Show EU table
    print("\n=== Expected Utility for each action (Go to location) ===")
    # Sort by EU descending for readability
    for L, val in sorted(eus.items(), key=lambda x: x[1], reverse=True):
        print(f"EU(Go{L}) = {val: .3f}")

    print("\n=== Best direct action ===")
    print(f"Best = Go{a_direct} with EU={eu_direct:.3f}")

    print("\n=== Information gathering (Value of Information) ===")
    print(f"EU({SENSE_ACTION}) = {eu_check:.3f}")
    print(f"VOI({SENSE_ACTION}) = EU(check) - EU(best_direct) = {voi:.3f}")

    print("\n=== Agent choice ===")
    print("Agent chooses:", choice)

    # Markdown-ready explanation 
    print("\n\n---\n### Markdown-ready explanation (paste into notebook)\n")
    print(f"""
**Uncertain variable (belief about the world):**
- For each location **L**, define **Busy(L, t)** as True when the fraction of all students at location L at time t exceeds {BUSY_THRESHOLD:.2f}.
- The belief state is \\( b(L,t) = P(Busy(L,t)=True) \\), estimated by Monte Carlo simulation from the Week 1 campus movement model.

**Actions:**
- Destination actions: Go to any location in the campus set (10 actions).
- Optional information action: **{SENSE_ACTION}** (checks whether StudentCenter is busy, with noisy sensor).

**Utility function (multi-attribute):**
- Utility is negative total expected “time cost” (minutes):
  - walking time + expected waiting time (depends on busyness) + mismatch penalty
- \\( U(GoL) = -[walk(type,L) + E[wait | Busy(L,t)] + mismatch(type,L)] \\)

**Probabilities assumed:**
- Beliefs \\(P(Busy(L,t))\\) come from Week 1 Monte Carlo traces at time **{DECISION_TIME}**.
- Sensor model for checking StudentCenter:
  - \\(P(ReportBusy | Busy)=0.85\\)
  - \\(P(ReportNotBusy | NotBusy)=0.85\\)

**Expected utility + rational choice:**
- For each destination action GoL, compute \\(EU(GoL)\\) using the belief \\(P(Busy(L,t))\\).
- For the info action, compute:
  - \\(EU(Check) = \\sum_o P(o) \\max_a EU(a|o) - cost\\)
- Choose the action with the highest expected utility.
""".strip())

=== Week 2 Decision @ 10:00 (time_step=1) for agent type='athlete' ===
Uncertain variable: Busy(L,t) defined as frac(students at L) > 0.18
Sense action: CheckStudentCenterCrowd with cost=2.0 (minutes)

=== Belief state (derived from Week 1 Monte Carlo) ===
P(Busy(LawrenceHall) at 10:00) ≈ 0.274
P(Busy(ThayerHall) at 10:00) ≈ 0.340
P(Busy(WestPenn) at 10:00) ≈ 0.086
P(Busy(StudentCenter) at 10:00) ≈ 0.531
P(Busy(Library) at 10:00) ≈ 0.251
P(Busy(VillagePark) at 10:00) ≈ 0.286
P(Busy(PlayHouse) at 10:00) ≈ 0.197
P(Busy(BoulevardStudios) at 10:00) ≈ 0.332
P(Busy(BoulevardAppartments) at 10:00) ≈ 0.086
P(Busy(OffCampus) at 10:00) ≈ 0.282

=== Expected Utility for each action (Go to location) ===
EU(GoVillagePark) = -8.858
EU(GoLibrary) = -13.255
EU(GoWestPenn) = -13.519
EU(GoBoulevardAppartments) = -13.519
EU(GoLawrenceHall) = -14.644
EU(GoThayerHall) = -15.037
EU(GoOffCampus) = -16.000
EU(GoPlayHouse) = -16.182
EU(GoStudentCenter) = -16.779
EU(GoBoulevardStudios) = -16.989

=== Best direc

## Results and Interpretation

The output of this notebook shows how a rational agent makes a decision under uncertainty using beliefs derived from simulation.

The model first estimates the probability that each campus location is busy at the decision time. Those probabilities are then used to compute the expected utility of each action. The agent compares direct movement actions with the alternative of paying for additional information before deciding.

This result demonstrates that sequential decision-making depends not only on what actions are available, but also on:
- uncertainty about the environment,
- the utility model used to evaluate outcomes,
- and the cost and usefulness of information.

## Connection to Sequential Decision-Making

This notebook illustrates a core idea in sequential decision-making: a rational agent must choose actions using both beliefs and preferences.

The agent:
- models uncertainty through probabilities,
- evaluates consequences through utility,
- and decides whether information has enough value to justify its cost.

This is a classic decision-theoretic framework. The agent is not simply reacting to the current state; it is reasoning about uncertainty, future outcomes, and whether an intermediate sensing action can improve long-run performance.

That is why this example connects directly to topics such as:
- Markov decision processes,
- partially observable decision-making,
- belief states,
- expected utility,
- and value of information.